# 05 — NER Evaluation with seqeval: Span-Level Metrics

**Objective:** evaluate the DNRTI NER model from notebook 04 *properly* — at the **span level** — and break the score down by entity type.

## Why not just reuse the training-time F1?

Notebook 04's training loop reported **token-level F1**: for every word in the evaluation set, is the predicted tag equal to the true tag? That number is easy to compute but *misleading for NER*.

**Why token-level is optimistic:**

Consider predicting on "Cobalt Strike" where the truth is `B-Tool  I-Tool`.
- Prediction `B-Tool  I-Tool` → token-level gets 2/2. Correct at span level: **✓**
- Prediction `B-Tool  B-Tool` → token-level gets 2/2 (both labels correct in isolation). Span level: **✗** — the model says there are two separate one-word tools instead of one two-word tool.
- Prediction `B-Tool  O`      → token-level gets 1/2. Span level: **✗** — the predicted span doesn't match the truth.

A real analyst only cares about the **span-level** answer: did the model find the *exact entity span* with the *right type*? That's what `seqeval` measures, and it's the number people report in NER papers.

## What this notebook does

1. Load the trained model from `models/ner_dnrti_final/` and the DNRTI test split
2. Run predictions → turn them back into BIO tag *strings* (seqeval needs strings, not IDs)
3. Compute **span-level** precision / recall / F1 (overall and per entity type)
4. Compare against the **token-level** number, to make the gap concrete
5. Look at error patterns: which entity types are hardest, where does the model confuse classes

## Prerequisite

Notebook 04 has run to completion and `models/ner_dnrti_final/` exists.

## Install

`seqeval` is separate from `sklearn.metrics`: `pip install seqeval`

## Step 1 — Load model, tokenizer, and test data

In [1]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import load_from_disk

MODEL_DIR = Path('models/ner_dnrti_final')
DATA_DIR = Path('processed/dnrti')

assert MODEL_DIR.exists(), f'Run notebook 04 first — {MODEL_DIR} missing.'
assert DATA_DIR.exists(), f'Run notebook 01 first — {DATA_DIR} missing.'

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()

ds = load_from_disk(str(DATA_DIR))
test = ds['test']
tag_names = test.features['ner_tags'].feature.names
id2tag = {i: t for i, t in enumerate(tag_names)}

print(f'Model loaded on {device}')
print(f'Test sentences: {len(test)}')
print(f'Tag classes: {len(tag_names)}  (first few: {tag_names[:6]})')

Model loaded on cuda
Test sentences: 664
Tag classes: 27  (first few: ['O', 'B-Area', 'B-Exp', 'B-Features', 'B-HackOrg', 'B-Idus'])


## Step 2 — Predict on the test set

We run the model sentence by sentence (simpler than batching, and the test set is small). For each sentence:

1. Tokenize the **word list** with `is_split_into_words=True`.
2. Get subword-level predictions from the model.
3. Collapse subword predictions back to **word-level** tags — take each word's first subword as the word's prediction.

That last step is critical: seqeval expects one tag per word, not one per subword. It also undoes the `-100` masking from training — we now care about every word, including ones we skipped during training.

In [2]:
MAX_LEN = 256


def predict_tags(words: list[str]) -> list[str]:
    """Return a list of predicted tag *strings*, one per input word."""
    enc = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt',
    )
    word_ids = tokenizer(words, is_split_into_words=True,
                         truncation=True, max_length=MAX_LEN).word_ids()

    with torch.no_grad():
        logits = model(**{k: v.to(device) for k, v in enc.items()}).logits[0]
    pred_ids = logits.argmax(-1).cpu().tolist()

    # Take the first subword's prediction for each word; default missing words to 'O'.
    out = ['O'] * len(words)
    seen = set()
    for wid, pid in zip(word_ids, pred_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        if wid < len(out):
            out[wid] = id2tag[pid]
    return out


# Collect predictions + ground truth as parallel lists-of-lists of strings.
y_true, y_pred = [], []
for ex in test:
    words = ex['tokens']
    truth = [id2tag[t] for t in ex['ner_tags']]
    pred = predict_tags(words)
    y_true.append(truth)
    y_pred.append(pred)

print(f'Predicted {len(y_pred)} sentences')
print(f'\nExample (first 10 tokens of test[0]):')
for w, t, p in list(zip(test[0]["tokens"], y_true[0], y_pred[0]))[:10]:
    mark = '' if t == p else '  <-- MISMATCH'
    print(f'  {w:20s}  truth={t:15s}  pred={p:15s}{mark}')

Predicted 664 sentences

Example (first 10 tokens of test[0]):
  Kaspersky             truth=B-SecTeam        pred=B-SecTeam      
  believes              truth=O                pred=O              
  both                  truth=O                pred=O              
  Shamoon               truth=B-HackOrg        pred=B-HackOrg      
  and                   truth=O                pred=O              
  StoneDrill            truth=B-HackOrg        pred=B-HackOrg      
  groups                truth=O                pred=I-HackOrg        <-- MISMATCH
  are                   truth=O                pred=O              
  aligned               truth=O                pred=O              
  in                    truth=O                pred=O              


## Step 3 — Span-level metrics with seqeval

`seqeval.metrics.classification_report` does two things `sklearn` can't:

1. It walks each predicted sequence looking for **spans**: `B-X` optionally followed by `I-X` runs. A span is counted correct only if both **boundary** and **type** match the truth.
2. It reports per-entity-type F1 plus micro/macro averages.

This is the number you cite externally.

In [3]:
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

p = precision_score(y_true, y_pred)
r = recall_score(y_true, y_pred)
f = f1_score(y_true, y_pred)

print(f'SPAN-LEVEL (micro-averaged):')
print(f'  precision: {p:.4f}')
print(f'  recall:    {r:.4f}')
print(f'  f1:        {f:.4f}')

print('\nPer-entity breakdown:\n')
print(classification_report(y_true, y_pred, digits=4))

SPAN-LEVEL (micro-averaged):
  precision: 0.7595
  recall:    0.8271
  f1:        0.7918

Per-entity breakdown:

              precision    recall  f1-score   support

        Area     0.8167    0.9074    0.8596       216
         Exp     0.9630    0.9848    0.9738       132
    Features     0.6753    0.8966    0.7704       116
     HackOrg     0.7543    0.8320    0.7912       369
        Idus     0.7883    0.8372    0.8120       129
      OffAct     0.7326    0.8400    0.7826       150
         Org     0.6552    0.6934    0.6738       137
        Purp     0.6084    0.7565    0.6744       115
     SamFile     0.8178    0.7782    0.7975       248
     SecTeam     0.8688    0.9145    0.8910       152
        Time     0.8779    0.8935    0.8856       169
        Tool     0.6609    0.7302    0.6938       315
         Way     0.7037    0.7600    0.7308       100

   micro avg     0.7595    0.8271    0.7918      2348
   macro avg     0.7633    0.8326    0.7951      2348
weighted avg     0.76

### How to read this

- Every row is an **entity type** (not a BIO tag). `seqeval` has already merged `B-Tool` + `I-Tool` into the `Tool` row.
- **support** = how many gold spans of that type exist in the test set. Low support = noisy F1 for that row.
- **micro avg** weights every span equally (dominated by frequent types).
- **macro avg** averages the per-type F1s (each type counts equally regardless of frequency).

The gap between micro and macro tells you how uneven performance is: a big gap means the model is good on common types and bad on rare ones.

## Step 4 — Token-level vs span-level: the gap

How much does the "correct number" differ from what notebook 04's training loop reported?

In [4]:
from sklearn.metrics import precision_recall_fscore_support

# Flatten word-level tags to IDs, ignore 'O' to make it comparable with typical NER reporting
def flatten_ids(seqs):
    return [tag_names.index(tag) for sent in seqs for tag in sent]

y_true_flat = flatten_ids(y_true)
y_pred_flat = flatten_ids(y_pred)

tp, tr, tf1, _ = precision_recall_fscore_support(
    y_true_flat, y_pred_flat, average='macro', zero_division=0
)

print(f'{"metric":12s}  {"token-level":>12s}  {"span-level":>12s}  gap')
print('-' * 52)
print(f'{"precision":12s}  {tp:12.4f}  {p:12.4f}  {p - tp:+.4f}')
print(f'{"recall":12s}  {tr:12.4f}  {r:12.4f}  {r - tr:+.4f}')
print(f'{"f1":12s}  {tf1:12.4f}  {f:12.4f}  {f - tf1:+.4f}')

metric         token-level    span-level  gap
----------------------------------------------------
precision           0.8121        0.7595  -0.0526
recall              0.8772        0.8271  -0.0501
f1                  0.8415        0.7918  -0.0496


Token-level F1 almost always looks better than span-level. A gap of 5–15 points is normal. If the gap is huge, the model is getting *tags* right but *boundaries* wrong — typically from hallucinating or merging spans.

## Step 5 — Where do the errors come from?

Two drill-downs that actually help:

1. **Confusion between entity types** — a matrix of `gold type → predicted type`. Reveals systematic mix-ups like Tool↔SamFile.
2. **Boundary errors vs type errors vs hallucinations** — three different failure modes, each with a different fix.

In [5]:
from collections import Counter


def extract_spans(tags: list[str]) -> list[tuple[int, int, str]]:
    """Return list of (start, end_inclusive, entity_type) spans from a BIO sequence."""
    spans = []
    start = None
    etype = None
    for i, t in enumerate(tags):
        if t.startswith('B-'):
            if start is not None:
                spans.append((start, i - 1, etype))
            start = i
            etype = t[2:]
        elif t.startswith('I-') and start is not None and t[2:] == etype:
            continue
        else:
            if start is not None:
                spans.append((start, i - 1, etype))
                start = None
                etype = None
    if start is not None:
        spans.append((start, len(tags) - 1, etype))
    return spans


# Classify each gold span as: exact_match / wrong_type / boundary_error / missed
# Classify each predicted span not in gold as: false_positive (hallucination) / wrong_type / boundary_error
exact = 0
wrong_type = 0
boundary = 0
missed = 0
hallucinated = 0
confusion = Counter()  # gold_type -> pred_type

for truth, pred in zip(y_true, y_pred):
    gold_spans = extract_spans(truth)
    pred_spans = extract_spans(pred)
    gold_set = set(gold_spans)
    pred_set = set(pred_spans)
    pred_by_range = {(s, e): t for s, e, t in pred_spans}

    # Build overlap index for boundary errors
    def any_overlap(s, e, spans):
        return [sp for sp in spans if not (sp[1] < s or sp[0] > e)]

    for gs, ge, gt in gold_spans:
        if (gs, ge, gt) in pred_set:
            exact += 1
        elif (gs, ge) in pred_by_range:
            wrong_type += 1
            confusion[(gt, pred_by_range[(gs, ge)])] += 1
        elif any_overlap(gs, ge, pred_spans):
            boundary += 1
        else:
            missed += 1

    for ps, pe, pt in pred_spans:
        if (ps, pe, pt) in gold_set:
            continue  # already counted as exact
        if (ps, pe) in {(gs, ge) for gs, ge, _ in gold_spans}:
            continue  # already counted as wrong_type
        if not any_overlap(ps, pe, gold_spans):
            hallucinated += 1

total_gold = sum(len(extract_spans(t)) for t in y_true)
print(f'Gold spans in test: {total_gold}')
print(f'  exact match:    {exact:5d}  ({exact/total_gold*100:.1f}%)')
print(f'  wrong type:     {wrong_type:5d}  ({wrong_type/total_gold*100:.1f}%)  -- right boundary, wrong class')
print(f'  boundary error: {boundary:5d}  ({boundary/total_gold*100:.1f}%)  -- overlaps but misaligned')
print(f'  missed:         {missed:5d}  ({missed/total_gold*100:.1f}%)  -- no overlapping prediction at all')
print(f'  hallucinated:   {hallucinated:5d}  (predicted spans with no gold overlap)')

if confusion:
    print('\nTop class confusions (gold -> predicted, same span):')
    for (gold, pred), n in confusion.most_common(10):
        print(f'  {gold:15s} -> {pred:15s}  {n}')

Gold spans in test: 2344
  exact match:     1929  (82.3%)
  wrong type:        89  (3.8%)  -- right boundary, wrong class
  boundary error:   171  (7.3%)  -- overlaps but misaligned
  missed:           155  (6.6%)  -- no overlapping prediction at all
  hallucinated:     253  (predicted spans with no gold overlap)

Top class confusions (gold -> predicted, same span):
  SamFile         -> Tool             18
  Tool            -> SamFile          14
  Tool            -> HackOrg          12
  Org             -> Idus             8
  HackOrg         -> Tool             6
  Purp            -> Features         3
  HackOrg         -> SecTeam          2
  Tool            -> SecTeam          2
  HackOrg         -> SamFile          2
  OffAct          -> HackOrg          2


### What the breakdown tells you

- **Exact match %** is essentially the recall number. High here = healthy model.
- **Wrong-type dominant** → the model finds entities but mislabels their class. Symptom of class-boundary ambiguity (e.g., `Tool` vs `SamFile` — is Cobalt Strike a tool or a sample?). Fix: cleaner labels, or a domain-pretrained model (notebook 10).
- **Boundary-error dominant** → the model knows an entity is there but can't pin down its edges. Common with noisy tokenization (punctuation, hyphens). Fix: pre-tokenization cleanup.
- **Missed dominant** → the model never detects the entity. Usually a rare class with too little training data.
- **Hallucinated dominant** → over-prediction. The loss is rewarding confidence too much — lower the learning rate or shorten training.

## Step 6 — A few concrete errors

Eyeball a handful of misclassified sentences. Numbers tell you *how much*; examples tell you *why*.

In [6]:
shown = 0
for idx, (truth, pred) in enumerate(zip(y_true, y_pred)):
    if truth == pred:
        continue
    gold_spans = set(extract_spans(truth))
    pred_spans = set(extract_spans(pred))
    if gold_spans == pred_spans:
        continue  # same spans, just different tag order — skip

    words = test[idx]['tokens']
    print(f'--- Sentence #{idx} ---')
    for w, t, p in zip(words, truth, pred):
        mark = '' if t == p else '  <--'
        print(f'  {w:25s}  truth={t:15s}  pred={p:15s}{mark}')
    print()
    shown += 1
    if shown >= 3:
        break

--- Sentence #0 ---
  Kaspersky                  truth=B-SecTeam        pred=B-SecTeam      
  believes                   truth=O                pred=O              
  both                       truth=O                pred=O              
  Shamoon                    truth=B-HackOrg        pred=B-HackOrg      
  and                        truth=O                pred=O              
  StoneDrill                 truth=B-HackOrg        pred=B-HackOrg      
  groups                     truth=O                pred=I-HackOrg        <--
  are                        truth=O                pred=O              
  aligned                    truth=O                pred=O              
  in                         truth=O                pred=O              
  their                      truth=O                pred=O              
  interests                  truth=O                pred=O              
  ,                          truth=O                pred=O              
  but                     

## Summary

| | Value |
|---|---|
| Model | DNRTI-fine-tuned BERT (from notebook 04) |
| Evaluation dataset | DNRTI test split |
| Metric | Span-level P/R/F1 via `seqeval` + per-entity breakdown |
| Diagnostics | Error-type breakdown + class confusion matrix |

### Takeaways

- Always report span-level F1 for NER. Token-level is fine for watching training progress but inflates the story.
- Rare entity types drag macro F1 down. If you care about them, collect more data for those classes before tuning hyperparameters.
- Error-type breakdown (exact / wrong-type / boundary / missed / hallucinated) tells you *what to fix next* far better than a single F1 number does.

**Next:** notebook 06 switches gears to the classification track — the math of multi-label classification (BCE loss, thresholds, Hamming loss) before notebook 07 trains it on TRAM.